# Qugeister - HNN Composer QNN学習

HNN Composerで設計したカスタムアーキテクチャを学習します。

## ワークフロー
1. QuAic HNN Composerで`config.json`をエクスポート
2. このノートブックで学習を実行
3. `weights.pth`をダウンロードしてQuAicに提出

**このノートブックはQugeister_cleanの動作済みコードを使用しています。**

## 1. 環境セットアップ

In [ ]:
# Qugeister_cleanをクローンしてセットアップ
import os
import shutil

# 既存のクローンを削除して最新版を取得
if os.path.exists('/content/Qugeister_clean'):
    shutil.rmtree('/content/Qugeister_clean')

!git clone --depth 1 https://github.com/ukinsama/Qugeister_clean.git /content/Qugeister_clean
!pip install -q pennylane torch numpy tqdm matplotlib

import sys
# srcディレクトリを直接パスに追加
sys.path.insert(0, '/content/Qugeister_clean/src')

# 必要なモジュールをインポート
import json
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
from google.colab import files

# Qugeister_cleanからインポート（srcを除いたパス）
from qugeister.models.hnn_config_loader import (
    load_hnn_config,
    build_model_from_config,
    HNNColorEstimator,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. 設定ファイルのアップロード

QuAic HNN Composerからエクスポートした`config.json`をアップロードしてください。

In [ ]:
# config.jsonをアップロード
print('config.jsonをアップロードしてください...')
uploaded = files.upload()

config_filename = list(uploaded.keys())[0]

# Qugeister_cleanのローダーを使用
hnn_config = load_hnn_config(config_filename)

print(f'\n=== HNN設定 ===')
print(f'量子ビット数: {hnn_config.quantum.n_qubits}')
print(f'量子レイヤー数: {hnn_config.quantum.n_layers}')
print(f'前処理レイヤー: {hnn_config.classical_pre}')
print(f'後処理レイヤー: {hnn_config.classical_post}')

## 3. モデルの構築

Qugeister_cleanの`build_model_from_config`を使用してモデルを構築します。

In [ ]:
# HNNColorEstimatorを使用（backpropモードを明示的に指定）
# backend='backprop'で500倍高速な勾配計算を使用
model = HNNColorEstimator(hnn_config, device='cpu', backend='backprop')

# 確認: backpropモードが有効か
# "[INFO] Forced backprop mode" と表示されれば高速モードが有効
print(f'\nパラメータ数: {sum(p.numel() for p in model.parameters()):,}')
print(f'\nstate_dict キー:')
for key, value in model.state_dict().items():
    print(f'  {key}: {value.shape}')

# GPUがあればモデルをGPUに移動（量子層以外）
# 注意: 量子層はCPUで実行され、結果がGPUに転送される
if torch.cuda.is_available():
    print(f'\n注意: GPUが利用可能ですが、量子層はCPUで実行されます')
    print('量子層の出力は自動的にGPUに転送されます')

## 4. 棋譜データのダウンロード

QuAicから棋譜データを自動ダウンロードします。

| データセット | ゲーム数 | 説明 |
|-------------|---------|------|
| `diverse_agents_3000` | 3000 | 5種のAIから収集（推奨） |
| `alphazero_mcts100_1000` | 1000 | AlphaZero高品質 |
| `alphazero_mcts100_5000` | 5000 | AlphaZero大規模 |

In [ ]:
# QuAicから棋譜データをダウンロード
import os
import urllib.request

QUAIC_BASE_URL = "https://quaic.up.railway.app"
DATASET_ID = "diverse_agents_3000"  # 変更可能: alphazero_mcts100_1000, alphazero_mcts100_5000

TRAJECTORY_FILE = f"{DATASET_ID}.pkl"

if not os.path.exists(TRAJECTORY_FILE):
    print(f'QuAicから {DATASET_ID} をダウンロード中...')
    url = f"{QUAIC_BASE_URL}/v1/competition/trajectories/{DATASET_ID}/download"
    urllib.request.urlretrieve(url, TRAJECTORY_FILE)
    print(f'ダウンロード完了: {TRAJECTORY_FILE}')
else:
    print(f'キャッシュを使用: {TRAJECTORY_FILE}')

# データを読み込み
with open(TRAJECTORY_FILE, 'rb') as f:
    trajectory_data = pickle.load(f)

print(f'読み込んだ試合数: {len(trajectory_data)}')

In [ ]:
def prepare_data(trajectory_data, train_ratio=0.8):
    """棋譜データから学習用データを準備
    
    Note: true_colorsに-1が含まれる場合は0に置換（捕獲済み駒）
    """
    X_list = []
    y_list = []

    for traj in trajectory_data:
        # Player A
        if 'states_A' in traj and 'true_colors_A' in traj:
            for state, colors in zip(traj['states_A'], traj['true_colors_A']):
                state = np.array(state).flatten()
                colors = np.array(colors)
                if state.shape[0] == 448 and colors.shape[0] == 8:
                    # -1を0に置換（捕獲済み駒は「良い駒」として扱う）
                    colors = np.clip(colors, 0, 1)
                    X_list.append(state)
                    y_list.append(colors)

        # Player B
        if 'states_B' in traj and 'true_colors_B' in traj:
            for state, colors in zip(traj['states_B'], traj['true_colors_B']):
                state = np.array(state).flatten()
                colors = np.array(colors)
                if state.shape[0] == 448 and colors.shape[0] == 8:
                    # -1を0に置換
                    colors = np.clip(colors, 0, 1)
                    X_list.append(state)
                    y_list.append(colors)

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int64)

    # シャッフル
    indices = np.random.permutation(len(X))
    X, y = X[indices], y[indices]

    # 分割
    split_idx = int(len(X) * train_ratio)
    X_train, X_val = X[:split_idx], X[split_idx:]
    y_train, y_val = y[:split_idx], y[split_idx:]

    print(f'学習データ: {len(X_train):,} サンプル')
    print(f'検証データ: {len(X_val):,} サンプル')

    return X_train, y_train, X_val, y_val

X_train, y_train, X_val, y_val = prepare_data(trajectory_data)

# DataLoader作成
train_dataset = TensorDataset(
    torch.tensor(X_train),
    torch.tensor(y_train)
)
val_dataset = TensorDataset(
    torch.tensor(X_val),
    torch.tensor(y_val)
)

# hnn_config.trainingはdict型（HNNConfig dataclass）
batch_size = hnn_config.training.get('batch_size', 32)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

print(f'\nバッチサイズ: {batch_size}')

## 5. 学習

In [ ]:
# 学習設定（hnn_config.trainingはdict型）
training_config = hnn_config.training
epochs = training_config.get('epochs', 50)
learning_rate = training_config.get('learning_rate', 0.001)

print(f'エポック数: {epochs}')
print(f'学習率: {learning_rate}')
print(f'バッチサイズ: {batch_size}')

# オプティマイザと損失関数
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    """1エポックの学習（CPU専用 - 量子層のため）"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch in loader:
        # 量子層はCPUで動作するのでデバイス移動は不要
        optimizer.zero_grad()
        outputs = model(X_batch)  # [batch, 8, 2]

        # 損失計算
        loss = criterion(outputs.view(-1, 2), y_batch.view(-1))

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # 精度計算
        preds = outputs.argmax(dim=-1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.numel()

    return total_loss / len(loader), correct / total


def validate(model, loader, criterion):
    """検証（CPU専用）"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in loader:
            outputs = model(X_batch)
            loss = criterion(outputs.view(-1, 2), y_batch.view(-1))

            total_loss += loss.item()

            preds = outputs.argmax(dim=-1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.numel()

    return total_loss / len(loader), correct / total

In [ ]:
# 学習ループ（CPU専用 - 量子層のbackpropはCPUで最速）
best_val_acc = 0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print('学習を開始します...')
print('注意: 量子層のbackprop微分はCPUで実行されます\n')

for epoch in tqdm(range(epochs), desc='Training'):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = validate(model, val_loader, criterion)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{epochs}')
        print(f'  Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}')
        print(f'  Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}')

print(f'\n学習完了! ベスト検証精度: {best_val_acc:.4f}')

## 6. 学習曲線の可視化

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# 損失
ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Validation')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.legend()
ax1.grid(True)

# 精度
ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'], label='Validation')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

## 7. モデルのエクスポート

学習済みモデルをダウンロードします。

**出力形式**: HNN形式 (`pre_layers.*`, `quantum_layer.weights`, `post_layers.*`)

In [ ]:
# ベストモデルを読み込み
model.load_state_dict(torch.load('best_model.pth', weights_only=True))

# エクスポート（HNN形式のまま）
# config_filenameから拡張子を除いてモデル名として使用
import os
model_name = os.path.splitext(config_filename)[0]
export_filename = f'{model_name}_weights.pth'

torch.save(model.state_dict(), export_filename)
print(f'モデルを保存しました: {export_filename}')
print(f'\nstate_dict キー:')
for key, value in model.state_dict().items():
    print(f'  {key}: {value.shape}')

In [ ]:
# ダウンロード
print('ダウンロードを開始...')
files.download(export_filename)
print('\nダウンロード完了!')
print('\n次のステップ:')
print('1. QuAic (https://quaic.up.railway.app) にアクセス')
print('2. 「コンペティション > モデル」に移動')
print('3. ダウンロードした .pth ファイルをアップロード')
print('\n※ HNN形式 (pre_layers.*, quantum_layer.*, post_layers.*) はQuAicで自動認識されます')